# From Model to Field
**Author:** Antoine ([@antowan](https://github.com/antowan))

## Engineering reliable edge AI applications on embedded GPUs

This notebook explains how a YOLO object detector becomes a working **AI Processor** (AIP) on an EdgeAI Station, and how to
choose between three models of the same family. It is the reading companion to the hands-on notebooks, and it has no code to run.

The running example is **AIPV1EMI0001**, a `car` / `person` / `truck` detector with one detector per package and three interchangeable
YOLO *flavors*:

| Flavor | Model | Hands-on build notebook |
|---|---|---|
| `yolov7` | YOLOv7-tiny (anchor-based, NMS inside the ONNX graph) | [`01-build-yolov7.ipynb`](01-build-yolov7.ipynb) |
| `yolov10` | YOLOv10n (NMS-free head) | [`02-build-yolov10.ipynb`](02-build-yolov10.ipynb) |
| `yolo26` | YOLO26n (NMS-free head) | [`03-build-yolo26.ipynb`](03-build-yolo26.ipynb) |

All three use the publishers' official COCO weights and report the same three classes. [`04-compare-models.ipynb`](04-compare-models.ipynb)
compares their speed and accuracy on the station. Every number shown here comes from a run recorded in this repository.
Neither the build notebooks nor this one train a model from scratch.

## Agenda

1. A model is not an application
2. The AI Processor: one contract from model to dashboard
3. One application, three models: flavors
4. The GPU video pipeline
5. From ONNX to a TensorRT engine
6. Choosing a model: accuracy and speed
7. Measuring fairly
8. Packaging reproducibly
9. Validating in layers
10. Operating in the field
11. Checklist

## 1. A model is not an application

A notebook demo answers *“does the model detect cars?”*. A field deployment must answer much more:

| In the lab | In the field |
|---|---|
| One video file, one GPU, one user | Up to 8 cameras sharing one embedded GPU and its memory with other services |
| Python loop around `model(frame)` | Hardware decode, batching, tracking, analytics, 24/7 |
| Latest framework from `pip` | Pinned SDK stack (JetPack, TensorRT, DeepStream) per device generation |
| “It ran once” | Repeatable build, verified package, acceptance test, versioned output contract |
| Results on screen | Business data delivered to applications and dashboards |

The rest of this notebook is the engineering that closes that gap.

## 2. The AI Processor: one contract from model to dashboard

An **AI Processor (AIP)** is a self-describing folder. The station assembles the video pipeline from the manifest and the configuration files:
the author supplies these declarative settings and the application components rather than writing the pipeline orchestrator.

```mermaid
flowchart LR
    subgraph AIP["AI Processor"]
        M["manifest.json<br/>what it is"]
        C["configs/<br/>how to infer and track"]
        MO["models/<br/>engine, labels, parser"]
        CB["callback.py<br/>what it outputs"]
        L["logic/ + dashboard/<br/>how it is used"]
    end
    AIP -- "eais-tools aip build / export" --> PKG[["verified .tgz"]]
    PKG --> ST["EdgeAI Station"]
    ST --> APP["Applications<br/>Node-RED, dashboards"]
```

**AIPV1EMI0001 at a glance** (from `manifest.json`):

| Field | Value |
|---|---|
| Identity | `AIPV1EMI0001` |
| Hardware targets | EX3, EX5, EX7 |
| Input streams | up to 8 |
| Detector | One of YOLOv7-tiny, YOLOv10n or YOLO26n, classes `car`, `person`, `truck`, batch 8 |
| Tracker | NvDCF, 640×384 |
| Analytics | ROI, LCC (region of interest, line crossing) |

## 3. One application, three models: flavors

Everything except the detector is identical across the three models: the manifest structure, the tracker, the callback, the Node-RED
logic and the dashboard. The detector is the only part that changes, so the repository keeps what *is* model-specific in one folder per flavor
and selects one with a single command:

```mermaid
flowchart LR
    subgraph F["flavors/"]
        Y7["yolov7/"]
        Y10["yolov10/"]
        Y26["yolo26/"]
    end
    F -- "tools/flavors.py use ID" --> W["manifest.json, store.json<br/>configs/ and config.pbtxt<br/>LICENSE.weights"]
    F -- "tools/export_model.py ID" --> O["ONNX model"]
    O -- "bin/build.sh ID" --> E["TensorRT engine + parser"]
    W --> P[["AIP package"]]
    E --> P
```

Each `flavor.json` records the official weights (URL and SHA-256), the pinned exporter, the engine name, the parser function, the
expected output tensors and the accuracy published by the model's authors, so nothing about a model is hidden in a script.

| | YOLOv7-tiny | YOLOv10n | YOLO26n |
|---|---|---|---|
| Output head | Anchors, then NMS **inside the ONNX graph** (`EfficientNMS_TRT` plugin) | **NMS-free**, 300 candidates | **NMS-free**, 300 candidates |
| Parser function | `NvDsInferParseCustomEfficientNMS` | `NvDsInferParseYoloEndToEnd` | `NvDsInferParseYoloEndToEnd` |
| Exporter | YOLOv7 repository at a pinned commit | `ultralytics` package, pinned | `ultralytics` package, pinned |
| Weights licence | GPL-3.0 | AGPL-3.0 | AGPL-3.0 |

## 4. The GPU video pipeline

```mermaid
flowchart LR
    SRC["Cameras<br/>RTSP / files"] --> DEC["Hardware decode"]
    DEC --> MUX["Batch frames<br/>from all streams"]
    MUX --> INF["Inference<br/>TensorRT engine"]
    INF --> TRK["Tracking<br/>nvtracker, NvDCF"]
    TRK --> ANA["Analytics<br/>ROI, line crossing"]
    ANA --> CB["callback.py"]
    CB --> NR["Node-RED"]
    NR --> UI["Dashboard"]
```

Design principles:

- **Keep frames in GPU-accessible buffers** where the DeepStream path supports it; avoid unnecessary CPU copies.
  CPU cores still handle orchestration, callbacks and other services.
- **Batch across cameras**: a batch lets inference process frames from several streams together; the benefit depends on the workload and the batch size.
- **In-process inference server** (`nvinferserver`): no network hop between video and model.
- **Track, then count**: associating detections across frames supports more stable counts and line-crossing analytics.
- **Callbacks consume metadata, not image tensors**: `callback.py` aggregates per-second counts; encoded HLS or WebRTC video follows a separate path to the dashboard.

## 5. From ONNX to a TensorRT engine

```mermaid
flowchart LR
    T["Training<br/>PyTorch"] --> O["ONNX<br/>portable model"]
    O -- "trtexec on the target device" --> E["TensorRT engine<br/>FP16, batch 1 to 8"]
    E --> D["DeepStream<br/>nvinferserver + parser"]
```

| Artifact | Portability | Role |
|---|---|---|
| ONNX | More portable than an engine; plugin nodes such as `EfficientNMS_TRT` still need support in the target TensorRT | Source for rebuilding and validation |
| TensorRT engine | **No**: tied to the TensorRT version, the GPU architecture and the build profile | What actually runs; always built on the target stack |

### The last mile: what the model outputs

An object detector first produces thousands of overlapping candidate boxes. Something has to keep the best one per object, and *where* that happens
is the main difference between the three flavors:

```mermaid
flowchart LR
    subgraph A["YOLOv7-tiny"]
        A1["network"] --> A2["EfficientNMS_TRT<br/>plugin in the engine"] --> A3["up to 100 detections"]
    end
    subgraph B["YOLOv10n and YOLO26n"]
        B1["network with<br/>one-to-one head"] --> B2["300 candidates<br/>no NMS needed"]
    end
    A3 --> P["C++ parser<br/>NvDsInferObjectDetectionInfo"]
    B2 --> P
    P --> N["nvinferserver<br/>score filter and clustering"]
```

A **custom parser** is the glue between a model and DeepStream: it reads the output tensors and returns detections in DeepStream's structure.
`src/nvdsinfer_customparser/` holds one function per output layout and maps the COCO classes to this AIP's classes
(`person` 0 → 1, `car` 2 → 0, `truck` 7 → 2).

### Every optimization is a claim

Pair each one with a **measurement** (did it help, and by how much?) and a **check** (does the result still work correctly?);
treat any claim without both as an assumption.

```mermaid
flowchart LR
    O["Optimize<br/>build choice"] --> M["Measure<br/>same method for every variant"]
    M --> V["Validate<br/>still loads, runs and detects"]
    V -- "slower, or wrong output" --> O
```

| Optimization | Measured by | Validated by |
|---|---|---|
| **TensorRT engine** built on the target | TensorRT FP32 against ONNX Runtime CUDA FP32 | Engine smoke test on the device |
| **FP16** | TensorRT FP16 against TensorRT FP32 | mAP of FP16 against FP32 on labelled images ([`04-compare-models.ipynb`](04-compare-models.ipynb)) |
| **Dynamic batch 1 / 8 / 8**: one engine for 1 to 8 streams | Latency at batch 1, 4 and 8 | Config validators; the same batch in engine, Triton, DeepStream and manifest |
| **Workspace 1024 MiB** (shared CPU/GPU memory) | A 4096 MiB build was OOM-killed on a running station | The build completes; the smoke test passes |

## 6. Choosing a model: accuracy and speed

### What “accuracy” means for a detector

A detection counts as correct when its box overlaps a ground-truth box of the same class by enough
**IoU** (intersection over union). How much is “enough” changes the score:

| Metric | Meaning |
|---|---|
| **AP 50** | Average precision when a detection must reach IoU ≥ 0.50 with the ground truth: forgiving about box placement |
| **mAP 50:95** | The average of AP over IoU thresholds 0.50, 0.55, … 0.95, then over classes: rewards **tight** boxes. This is the **COCO primary metric** |

**mAP 50:95 is what Ultralytics publishes** in the model tables of its documentation (the *mAP<sup>val</sup> 50-95* column, COCO val2017, 640 px input; for YOLO26 use the *(e2e)* column, which matches the NMS-free head deployed here).
The evaluation in this repository (`tools/accuracy.py`) uses the same COCO protocol, `pycocotools`, on the same 5,000 images, so its numbers
can be compared to the published ones. As a check, the evaluator reproduces the published figures for YOLO26n and YOLOv10n to within a few tenths of a point. YOLOv7-tiny has no comparable official figure.

### Two sets of classes

The official weights are trained on the 80 COCO classes, and the AIP reports three. The tables therefore give both views:

- **80 classes**: the standard COCO benchmark, directly comparable with the published numbers.
- **car, person, truck**: the same engine scored only on the classes this AIP uses (the other detections are ignored). It comes out a little higher
  here, mostly because `person` is detected better than most COCO classes.

### Results on the station

Measured on an Orin NX 16 GB with TensorRT FP16: accuracy over the 5,000 COCO val2017 images, and latency in a separate synthetic-input run (the full tables are in [`doc/benchmarks/orin-nx-16gb-jp6.0/comparison.md`](../doc/benchmarks/orin-nx-16gb-jp6.0/comparison.md)):

| Model | mAP 50:95, 80 classes (published) | mAP 50:95, car / person / truck | FP16 ms per image, batch 1 | FP16 ms per image, batch 8 |
|---|---:|---:|---:|---:|
| YOLO26n | 39.92 (40.1) | 40.67 | 5.3 | 3.4 |
| YOLOv10n | 38.34 (38.5) | 40.54 | 5.5 | 3.8 |
| YOLOv7-tiny | 37.13 (n/a) | 39.41 | 8.4 | 4.7 |

![Latency and accuracy of the three flavors](../doc/benchmarks/orin-nx-16gb-jp6.0/comparison.png)

The latencies are raw model times, not DeepStream pipeline throughput. They were remeasured on 2026-11-18 with the video pipeline stopped and freshly rebuilt engines. The accuracy column retains the earlier evaluation; the rebuilt engines were not rescored. See the [measurement conditions and previous timings](../doc/benchmarks/orin-nx-16gb-jp6.0/conditions.md).

### Reading the results

- **Accuracy closely agrees with the published references.** The measured 80-class mAP 50:95 is within 0.2 points of the published figures for YOLO26n (40.1, the Ultralytics end-to-end column) and YOLOv10n (38.5, from the YOLOv10 paper), measured on the station's TensorRT FP16 engines. This is close agreement, not an exact reproduction of the authors' inference setup or a validation of the DeepStream pipeline. Ultralytics publishes 4.13 ms for YOLO26n FP16 on an Orin NX 16GB, but matching runtime, load and timing conditions have not been established. Its Jetson guide uses COCO128 for accuracy, not full COCO val2017; see the [measurement conditions](../doc/benchmarks/orin-nx-16gb-jp6.0/conditions.md).
- **YOLO26n has the highest recorded accuracy and the lowest measured latency of the three.** It scores 39.9 against 38.3 for YOLOv10n and 37.1 for YOLOv7-tiny. In the idle latency rerun it needs 5.3 ms per image at batch 1 against 5.5 and 8.4. YOLOv7-tiny takes 1.6x as long as YOLO26n at batch 1 and 1.4x at batch 8. YOLOv10n sits close to YOLO26n in speed (3.8 against 3.4 ms at batch 8).
- **The three classes score a little higher than the 80** (40.7 against 39.9 for YOLO26n), mostly thanks to `person`: per class, YOLO26n reaches 52.7 on `person`, 37.0 on `car` and 32.3 on `truck`. Trucks are the hardest of the three and the least frequent in the validation set (414 annotations against 1,918 cars and 10,777 persons).
- **The earlier accuracy run found negligible FP16 loss.** Scoring its FP32 and FP16 engines on the same images differs by at most 0.02 points. In the separate idle latency rerun, FP16 is 2.9 to 3.6 times faster than ONNX Runtime with FP32 at batch 1, depending on the model.
- **Batching amortizes the cost.** YOLO26n needs 3.4 ms per image at batch 8 against 5.3 ms at batch 1, so a batch of frames, one per stream, is cheaper per image than one frame at a time.
- **These are idle-baseline measurements, not pipeline throughput.** All 100 GPU-load baseline samples were 0%, with power mode and clock settings unchanged. Alternating-engine GPU-only checks kept the same ordering. Batch-1 latency is 30 to 38% lower than in the earlier contended run, but the engines were also rebuilt, so the difference cannot be attributed solely to stopping the pipeline; see the [conditions](../doc/benchmarks/orin-nx-16gb-jp6.0/conditions.md).

### Why the earlier 12.6% was low

Earlier versions of this repository scored **12.64% mAP 50:95** on COCO val2017 and the evaluation looked suspicious. The evaluator was not the problem:
it reproduces the published numbers of the official models. The weights were: the model had been fine-tuned for three classes on other data,
and such weights transfer poorly to the COCO benchmark. Starting from the publishers' official COCO weights fixes the benchmark.
It does not prove accuracy on *your* cameras: always check on labelled images of the target scene.

## 7. Measuring fairly

*“TensorRT is N× faster than ONNX”* is meaningless unless both sides are measured the same way. Common traps:

- `trtexec` reports GPU compute time and **pipelines** requests; a Python ONNX Runtime loop includes dispatch and synchronization. Different boundaries, different numbers.
- ONNX Runtime's TensorRT provider builds **its own** engine: it does not measure yours.
- TF32 enabled on one side only; different batch sizes, input copies inside the timer, cold start.

The method used by [`tools/benchmark.py`](../tools/benchmark.py) (following
[Ultralytics' predictor](https://github.com/ultralytics/ultralytics/blob/80c34c050921abd29970d8e3d2a2a41ec6538c98/ultralytics/engine/predictor.py#L380-L397)):

| Control | Method |
|---|---|
| Graph | The same model for every runtime, without NMS (for YOLOv7 the NMS node is cut off) |
| Precision flags | ONNX Runtime CUDA uses `use_tf32=0`; TensorRT FP32 uses `--noTF32`; TensorRT FP16 adds `--fp16` |
| Data | One GPU-resident input per batch; preallocated GPU outputs |
| Timer | `cudaDeviceSynchronize → perf_counter → one call → cudaDeviceSynchronize` |
| Statistics | 20 warm-up + 50 timed requests × 3 rounds, shuffled order, median and p95 |
| Placement | Profile untimed ONNX Runtime requests on the same session, then stop profiling |
| Conditions | Versions, file hashes and GPU load, clock and temperature around each round |

Two measurements answer two different questions; do not mix them:

- **Raw model latency**: milliseconds per inference request, one at a time, without NMS. It answers *“how fast does this runtime execute the model on this station?”*
- **Pipeline throughput** (`eais-tools check perf`): frames per second across all streams, including decode, tracking and callback.
  It answers *“how many cameras can this station handle?”*

The latency tables above are raw model latency. Video pipelines were stopped during the runs, while the other station services stayed up.

## 8. Packaging reproducibly

```mermaid
flowchart LR
    SRC["Repository<br/>+ built engine and parser"] --> B["eais-tools aip build<br/>--include-file .include"]
    B --> D["dist/"]
    D --> X["eais-tools aip export"]
    X --> A[["AIPV1EMI0001_dist.tgz"]]
    A -. "release CI" .-> CK["Additional file-set<br/>and archive integrity checks"]
```

- **Native packaging**: EAIS Tools builds and exports the runtime package. `.include` adds files beyond its defaults; no second builder is needed.
- **Explicit additions**: this repository's `.include` names individual files. Directories would also ship `node_modules/`, caches or `flows_cred.json` (credentials).
- **One flavor per package**: `tools/flavors.py check` and `tools/check_package.py` fail if the manifest, the configuration, the licence or the engine
  belong to different flavors.
- **Weights outside Git**: official weights are downloaded from their publisher and verified by SHA-256 before export. Engines are rebuilt on the target stack.

## 9. Validating in layers

```mermaid
flowchart LR
    S["Static<br/>eais-tools aip check<br/>Triton / DeepStream validators"] --> E["Engine smoke test<br/>loads and runs on this device"]
    E --> L["Labelled model / engine<br/>mAP + per-class AP"]
    E --> P["Pipeline test<br/>eais-tools pipeline test<br/>decode, infer, track, callback"]
    P --> C["Output contract<br/>schema + detections > 0"]
    C --> U["Dashboard<br/>human check"]
```

- Static checks are fast but cannot run the tracker or the GPU.
- The **station-managed pipeline test** runs the real pipeline headless, on a sample video.
- **A started pipeline, or a timeout, is not a pass**: the test requires valid callback messages *with* detections.
- **Labelled accuracy** is a different check from latency or a pipeline smoke test. It uses the raw engines on COCO images and does not exercise DeepStream parsing, tracking or the callback.
- The callback output is a documented, versioned **contract** (`doc/USER_MANUAL.md`), checked automatically: applications depend on it, not on the model.
- **Know what is not proven yet**: accuracy on the target scenes (labelled images), capacity in streams at the target frame rate (`eais-tools check perf`) and 24/7 stability (soak test).

## 10. Operating in the field

- **Business logic as a Node-RED flow** shipped with the AIP (`logic/`): subscribes to the pipeline's callback and normalizes counts per class, region of interest and line crossing.
- **Dashboard shipped with the AIP** (`dashboard/`, Vue 3 UIBuilder): pipeline-selected HLS or WebRTC video and counts; browser libraries are vendored so it works **offline**.
- **No secrets in packages**: `flows_cred.json` never ships; credentials are configured on the station.
- **One package, many stations**: install the verified `.tgz`; rebuild the engine from ONNX when the SDK generation changes.
- **Watch the device, not just the model**: memory headroom (shared CPU/GPU), GPU load and temperature, stream health.

## 11. Checklist: from model to field

- [ ] Start from the publisher's official weights and pin their checksum
- [ ] Export a portable model (ONNX) with a pinned exporter and check its inputs and outputs against the configuration
- [ ] Build the engine **on** the target SDK stack
- [ ] Choose precision and batch profile deliberately, and size builder memory for a shared device
- [ ] Benchmark with **one** timing boundary for every backend; report conditions and medians
- [ ] Evaluate labelled accuracy with the metric the model's authors publish, and compare FP16 with FP32
- [ ] Package with an allowlist and verify the exact package contents
- [ ] Validate statically, then on the real pipeline, then the output contract
- [ ] Ship the application logic and the dashboard with the model, without secrets
- [ ] Make the whole path reproducible by someone else: notebooks and a CI workflow

## Try it yourself

1. Pick a flavor and run its build notebook on a station: [`01-build-yolov7.ipynb`](01-build-yolov7.ipynb), [`02-build-yolov10.ipynb`](02-build-yolov10.ipynb) or [`03-build-yolo26.ipynb`](03-build-yolo26.ipynb).
2. Compare the models, or measure them again on your own station: [`04-compare-models.ipynb`](04-compare-models.ipynb).
3. Install the dashboard with `bin/install-nodered-project.sh` (see [`logic/README.md`](../logic/README.md)).

## Resources

- Source, notebooks and dashboard: [github.com/edge-ai/AIPV1EMI0001](https://github.com/edge-ai/AIPV1EMI0001)
- Benchmark and accuracy tools: [`tools/benchmark.py`](../tools/benchmark.py), [`tools/accuracy.py`](../tools/accuracy.py), [`tools/compare_models.py`](../tools/compare_models.py),
  and the recorded results in [`doc/benchmarks/`](../doc/benchmarks/)
- YOLOv7: [WongKinYiu/yolov7](https://github.com/WongKinYiu/yolov7)
- YOLOv10 and YOLO26: [Ultralytics documentation](https://docs.ultralytics.com/models/)